In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import pickle
import os

In [10]:
df = pd.read_csv('data/sleep_data.csv')

print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nRisk label distribution:")
print(df['risk_label'].value_counts())


Shape: (70, 6)

First 5 rows:
   total_screen_time  unlock_count  longest_session  hour_of_first_use  \
0                 55            14               30                 23   
1                 48            12               25                 23   
2                 62            16               35                  0   
3                 70            18               40                  1   
4                 45            11               22                 23   

   app_category risk_label  
0             2       High  
1             3       High  
2             2       High  
3             2       High  
4             3       High  

Risk label distribution:
risk_label
Medium    25
Low       25
High      20
Name: count, dtype: int64


In [11]:
# Features (inputs to model)
X = df[[
    'total_screen_time',
    'unlock_count',
    'longest_session',
    'hour_of_first_use',
    'app_category'
]]

# Label (what we want to predict)
y = df['risk_label']

print("Features shape:", X.shape)
print("Labels:", y.unique())

Features shape: (70, 5)
Labels: ['High' 'Medium' 'Low']


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42,    # same split every time
    stratify=y          # keep class balance
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 56
Testing samples: 14


In [13]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully")
print("Classes:", model.classes_)

Model trained successfully
Classes: ['High' 'Low' 'Medium']


In [14]:
y_pred = model.predict(X_test)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

=== Classification Report ===
              precision    recall  f1-score   support

        High       1.00      1.00      1.00         4
         Low       1.00      1.00      1.00         5
      Medium       1.00      1.00      1.00         5

    accuracy                           1.00        14
   macro avg       1.00      1.00      1.00        14
weighted avg       1.00      1.00      1.00        14

=== Confusion Matrix ===
[[4 0 0]
 [0 5 0]
 [0 0 5]]


In [15]:
# Simulate a high risk night
test_input = [[47, 13, 22, 23, 2]]
prediction = model.predict(test_input)[0]
print("Test prediction:", prediction)  # Expected: High

# Simulate a low risk night
test_input2 = [[8, 2, 5, 22, 0]]
prediction2 = model.predict(test_input2)[0]
print("Test prediction 2:", prediction2)  # Expected: Low

Test prediction: High
Test prediction 2: Low


d:\sleep_assistant\venv\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
d:\sleep_assistant\venv\lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [16]:
os.makedirs('models', exist_ok=True)

with open('models/sleep_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model saved to models/sleep_model.pkl")

# Also save a copy for cloud functions
os.makedirs('cloud_functions', exist_ok=True)
with open('cloud_functions/model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Model copy saved to cloud_functions/model.pkl")

Model saved to models/sleep_model.pkl
Model copy saved to cloud_functions/model.pkl
